In [1]:
from ingest import load_faq_data

documents=load_faq_data()

In [2]:
documents[19]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Besides the “Office Hour” which are the live zoom calls?',
 'answer': 'We will probably have some calls during the Capstone period to clear some questions, but it will be announced in advance if that happens.\n\nSee [Google Calendar](https://calendar.google.com/calendar/?cid=ZXIxcjA1M3ZlYjJpcXU0dTFmaG02MzVxMG9AZ3JvdXAuY2FsZW5kYXIuZ29vZ2xlLmNvbQ)',
 'doc_id': '99bb0ceeb6'}

In [3]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

103

In [4]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [5]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [6]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client=OpenAI()

In [7]:
doc=documents[0]

In [8]:
import json
user_prompt = json.dumps(doc)

#user_prompt

In [9]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [10]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [11]:
result = response.output_parsed

result.questions

['When does the next Data Engineering Zoomcamp cohort usually begin, and where can I find the exact start date?',
 'How do I register for the current Zoomcamp cohort before it starts?',
 'Is the course start date the same every year, or does it change by cohort?',
 'Where should I check for the official registration link and latest course announcements?',
 'Which community channels should I join to stay updated about the course launch?']

In [12]:
from evaluation_utils import llm_structured

In [13]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['When does the next data engineering zoomcamp cohort usually begin?', 'Where can I find the exact start date for this year’s cohort?', 'How do I register for the course before it starts?', 'Is there a place to get announcements about the zoomcamp start date?', 'Do I need to join any Telegram or Slack channels for course updates?']


In [14]:
result.questions

['When does the next data engineering zoomcamp cohort usually begin?',
 'Where can I find the exact start date for this year’s cohort?',
 'How do I register for the course before it starts?',
 'Is there a place to get announcements about the zoomcamp start date?',
 'Do I need to join any Telegram or Slack channels for course updates?']

In [15]:
result

Questions(questions=['When does the next data engineering zoomcamp cohort usually begin?', 'Where can I find the exact start date for this year’s cohort?', 'How do I register for the course before it starts?', 'Is there a place to get announcements about the zoomcamp start date?', 'Do I need to join any Telegram or Slack channels for course updates?'])

In [16]:
from evaluation_utils import calc_price

In [17]:
cost = calc_price(usage)

cost

{'input_cost': 0.00022275000000000002,
 'output_cost': 0.0003645,
 'total_cost': 0.0005872500000000001}

In [20]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["doc_id"]
    })

records

[{'question': 'When does the next data engineering zoomcamp cohort usually begin?',
  'document': '9e508f2212'},
 {'question': 'Where can I find the exact start date for this year’s cohort?',
  'document': '9e508f2212'},
 {'question': 'How do I register for the course before it starts?',
  'document': '9e508f2212'},
 {'question': 'Is there a place to get announcements about the zoomcamp start date?',
  'document': '9e508f2212'},
 {'question': 'Do I need to join any Telegram or Slack channels for course updates?',
  'document': '9e508f2212'}]

In [21]:
import pandas as pd

In [23]:
pd.DataFrame(records)

,question,document
0,When does the next data engineering zoomcamp c...,9e508f2212
1,Where can I find the exact start date for this...,9e508f2212
2,How do I register for the course before it sta...,9e508f2212
3,Is there a place to get announcements about th...,9e508f2212
4,Do I need to join any Telegram or Slack channe...,9e508f2212


In [24]:
from evaluation_utils import llm_structured_retry

In [25]:
def generate_ground_truth(doc):
    user_prompt=json.dumps(doc)

    out, usage= llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results=[]

    for q in out.questions:
        results.append({
            "question":q,
            "document":doc["doc_id"]
        })
    return results, usage

In [26]:
generate_ground_truth(doc)

([{'question': 'When does the next data engineering cohort actually start?',
   'document': '9e508f2212'},
  {'question': 'Is the course only available in a certain part of the year?',
   'document': '9e508f2212'},
  {'question': 'Where can I find the exact start date for the current cohort?',
   'document': '9e508f2212'},
  {'question': 'Do I need to register somewhere before the course begins?',
   'document': '9e508f2212'},
  {'question': 'What’s the best place to follow announcements for the course start?',
   'document': '9e508f2212'}],
 ResponseUsage(input_tokens=297, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=76, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=373))

In [27]:
from tqdm.auto import tqdm

gr_truth=[]
usages=[]

for i in tqdm(documents[:5]):
    records, usage = generate_ground_truth(i)
    gr_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [28]:
usages

[ResponseUsage(input_tokens=297, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=82, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=379),
 ResponseUsage(input_tokens=324, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=109, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=433),
 ResponseUsage(input_tokens=231, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=105, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=336),
 ResponseUsage(input_tokens=223, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=105, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=328),
 ResponseUsage(input_tokens=287, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=88, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=375)]

In [ ]:
gr_truth[:5]

NameError: name 'gr_truth' is not defined

In [33]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress